# YOLOv8 Surface Defect Detection Training
MVTec AD Dataset | Vision Inspection Portfolio

In [ ]:
!pip install ultralytics -q

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path
import subprocess

bottle_dir = Path('/content/drive/MyDrive/vision_portfolio/bottle')
if bottle_dir.exists():
    shutil.rmtree(bottle_dir)
    print("Old folder deleted")

result = subprocess.run([
    'unzip',
    '/content/drive/MyDrive/vision_portfolio/bottle_yolo.zip',
    '-d', '/content/drive/MyDrive/vision_portfolio/'
], capture_output=True, text=True)
print("Extraction complete")

In [ ]:
import yaml

yaml_path = '/content/drive/MyDrive/vision_portfolio/bottle/dataset.yaml'

data = {
    'path': '/content/drive/MyDrive/vision_portfolio/bottle',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,
    'names': {0: 'broken_small', 1: 'broken_large', 2: 'contamination'}
}

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("dataset.yaml updated")
with open(yaml_path, 'r') as f:
    print(f.read())

In [ ]:
from pathlib import Path

base = Path('/content/drive/MyDrive/vision_portfolio/bottle')
train_images = list((base / 'images/train').glob('*.*'))
val_images = list((base / 'images/val').glob('*.*'))
train_labels = list((base / 'labels/train').glob('*.txt'))
val_labels = list((base / 'labels/val').glob('*.txt'))

empty_train = sum(1 for f in train_labels if f.stat().st_size == 0)
empty_val = sum(1 for f in val_labels if f.stat().st_size == 0)

print(f"Train images: {len(train_images)}")
print(f"Val images: {len(val_images)}")
print(f"Train labels (empty/total): {empty_train}/{len(train_labels)}")
print(f"Val labels (empty/total): {empty_val}/{len(val_labels)}")

assert len(train_images) > 0, "No train images found"
assert empty_train < len(train_labels), "All train labels are empty - check conversion"
print("Dataset verification PASSED")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/drive/MyDrive/vision_portfolio/bottle/dataset.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    name='bottle_test',
    project='/content/drive/MyDrive/vision_portfolio/runs',
    patience=5,
    save=True,
    plots=True,
    exist_ok=True
)

print(f"\nBest mAP50: {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

run_dir = Path('/content/drive/MyDrive/vision_portfolio/runs/bottle_test')

for metric in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    img_path = run_dir / metric
    if img_path.exists():
        img = plt.imread(str(img_path))
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(metric)
        plt.axis('off')
        plt.show()

In [ ]:
from ultralytics import YOLO

best_pt = '/content/drive/MyDrive/vision_portfolio/runs/bottle_test/weights/best.pt'
model = YOLO(best_pt)
model.export(format='onnx', imgsz=640, simplify=True)
print("ONNX export complete")
print(f"Saved to: {best_pt.replace('.pt', '.onnx')}")